### Notebook *NB04d – Modelo LSTM CON  FULL sentimiento (horizonte 5 días)*
**Autor:** Jesús Daniel Romeral Cortina

**Objetivo:**  
Entrenar y evaluar modelo LSTM utilizando variables financieras del S&P 500 junto con un conjunto ampliado de variables de sentimiento extraídas de noticias financieras (medianas, desviación típica y ratios), con el fin de analizar si la incorporación de sentimiento mejora la capacidad predictiva en la predicción direccional a 5 días frente al baseline sin sentimiento y a los modelos con sentmiento simplificado.

**Metodología (resumen):**
- Se aumentan las features de sentimiento a partir del dataset `sp500_sent_FULL.csv`.
- **Split temporal**: train anterior a **2022-01-01** y test posterior.
- Preparación de datos para LSTM mediante creación de secuencias con **lookback** fijo.
- Validación temporal interna: el último 20% del conjunto de entrenamiento se usa como validación.
- Las neuronas selecionadas son a partir de la mejor metrica sacada del modelo lstm 1d sin sentimiento para una compaación justa, manteniendo el resto de parámetros iguales.
- Métrica principal: **ROC_AUC** (además de Acc, B_Acc, Precision, Recall y F1).
- Métricas guardadas en `resultados_ml_5d_SENT_FULL.csv`.


In [1]:
import numpy as np
import pandas as pd
import os
import random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, roc_auc_score, precision_score, recall_score)
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers

2026-02-20 11:11:10.353503: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:

FAMILIA = "DL"
MODELO_BASE = "LSTM" 
HORIZONTE = "5d" 
SENTIMIENTO = "FULL"


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
MODEL_PATH   = "../../datos/sp500_sent_FULL.csv"
OUT_DIR = "../../resultados"
OUT_PATH = "../../resultados/resultados_lstm_5d_SENT_FULL.csv"

In [5]:
df = pd.read_csv(MODEL_PATH, parse_dates=["Date"])
df = df.sort_values("Date").set_index("Date")
df.head()

,Close,High,Low,Open,Volume,Return,Target_1d,Return_5d_forward,Target_5d,ret_lag_1,...,sentiment_mean,sentiment_median,sentiment_std,n_news,pos_ratio,neg_ratio,neu_ratio,has_news,sentiment_filled,n_news_filled
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-02,1462.420044,1462.430054,1426.189941,1426.189941,4202600000,0.025403,0,-0.000957,0,0.016942,...,0.470913,0.448076,0.545041,4.0,0.500000,0.0,0.500000,1,0.470913,4
2013-01-03,1459.369995,1465.469971,1455.530029,1462.420044,3829730000,-0.002086,1,0.008737,1,0.025403,...,0.000000,0.000000,0.000000,2.0,0.000000,0.0,1.000000,1,0.000000,2
2013-01-04,1466.469971,1467.939941,1458.989990,1459.369995,3424290000,0.004865,0,0.003805,1,-0.002086,...,0.000000,0.000000,0.000000,1.0,0.000000,0.0,1.000000,1,0.000000,1
2013-01-07,1461.890015,1466.469971,1456.619995,1466.469971,3304970000,-0.003123,0,0.006013,1,0.004865,...,0.000000,0.000000,0.000000,2.0,0.000000,0.0,1.000000,1,0.000000,2
2013-01-08,1457.150024,1461.890015,1451.640015,1461.890015,3601600000,-0.003242,1,0.010424,1,-0.003123,...,0.244576,0.000000,0.423618,3.0,0.333333,0.0,0.666667,1,0.244576,3


In [6]:
print("Shape:", df.shape)
print("Rango fechas:", df.index.min(), "-", df.index.max())
df.head(3)

Shape: (2811, 28)
Rango fechas: 2013-01-02 00:00:00 - 2024-03-04 00:00:00


,Close,High,Low,Open,Volume,Return,Target_1d,Return_5d_forward,Target_5d,ret_lag_1,...,sentiment_mean,sentiment_median,sentiment_std,n_news,pos_ratio,neg_ratio,neu_ratio,has_news,sentiment_filled,n_news_filled
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-02,1462.420044,1462.430054,1426.189941,1426.189941,4202600000,0.025403,0,-0.000957,0,0.016942,...,0.470913,0.448076,0.545041,4.0,0.5,0.0,0.5,1,0.470913,4
2013-01-03,1459.369995,1465.469971,1455.530029,1462.420044,3829730000,-0.002086,1,0.008737,1,0.025403,...,0.000000,0.000000,0.000000,2.0,0.0,0.0,1.0,1,0.000000,2
2013-01-04,1466.469971,1467.939941,1458.989990,1459.369995,3424290000,0.004865,0,0.003805,1,-0.002086,...,0.000000,0.000000,0.000000,1.0,0.0,0.0,1.0,1,0.000000,1


**Target (5 días)**
- *Target_5d* es la etiqueta binaria a predecir.
- Se elimina features *forward* para evitar fuga de información.

In [7]:
Y = df["Target_5d"]
X = df.drop(columns=[
    "Target_1d", 
    "Target_5d", 
    "Return_5d_forward",
    "Close",
    "High",
    "Low",
    "Open",
    "Volume",
    "sentiment_mean", 
    "n_news"
]) 
X.head(3)

,Return,ret_lag_1,ret_lag_2,ret_lag_3,ret_lag_4,ret_lag_5,ret_ma_5,ret_std_5,ret_ma_10,ret_std_10,sentiment_median,sentiment_std,pos_ratio,neg_ratio,neu_ratio,has_news,sentiment_filled,n_news_filled
Date,,,,,,,,,,,,,,,,,,
2013-01-02,0.025403,0.016942,-0.011050,-0.001218,-0.004787,-0.002440,0.005058,0.015419,0.002286,0.012203,0.448076,0.545041,0.5,0.0,0.5,1,0.470913,4
2013-01-03,-0.002086,0.025403,0.016942,-0.011050,-0.001218,-0.004787,0.005598,0.015030,0.000928,0.011814,0.000000,0.000000,0.0,0.0,1.0,1,0.000000,2
2013-01-04,0.004865,-0.002086,0.025403,0.016942,-0.011050,-0.001218,0.006815,0.014580,0.002174,0.011468,0.000000,0.000000,0.0,0.0,1.0,1,0.000000,1


**Split temporal por fecha (train/test)** 
Con embargo por horizonte

In [8]:
H = 5
train_mask = df.index < "2022-01-01"

X_train_raw = X.loc[train_mask].iloc[:-H]
y_train     = Y.loc[train_mask].iloc[:-H]

X_test_raw  = X.loc[~train_mask]
y_test      = Y.loc[~train_mask]

assert X_train_raw.index.max() < X_test_raw.index.min()
assert (X_train_raw.index == y_train.index).all()


print("Train:", X_train_raw.shape, "Test:",X_test_raw.shape)

pos_train = y_train.sum()
pos_test = y_test.sum()
print("Porcentaje positivos en train:", round(pos_train/ len(y_train) * 100, 1),"%")
print("Porcentaje positivos en test:", round(pos_test / len(y_test) * 100, 1),"%")



Train: (2262, 18) Test: (544, 18)
Porcentaje positivos en train: 62.2 %
Porcentaje positivos en test: 55.7 %


**Split temporal por fechas (train/val)** EN RAW + gap=H (embargo entre train y val)

In [9]:


val_size_raw = int(len(X_train_raw) * 0.2)
val_start_raw = len(X_train_raw) - val_size_raw

train_end_raw = max(0, val_start_raw - H)

X_tr_raw = X_train_raw.iloc[:train_end_raw]
y_tr = y_train.iloc[:train_end_raw]

X_val_raw = X_train_raw.iloc[val_start_raw:]
y_val = y_train.iloc[val_start_raw:]

In [10]:
print("TRAIN:", X_tr_raw.index[0],"-",X_tr_raw.index[-1],"n=", len(X_tr_raw))
print("VAL  :", X_val_raw.index[0],"-",X_val_raw.index[-1],  "n=", len(X_val_raw))
print("TEST :", X_test_raw.index[0],"-",X_test_raw.index[-1],"n=", len(X_test_raw))


TRAIN: 2013-01-02 00:00:00 - 2020-03-04 00:00:00 n= 1805
VAL  : 2020-03-12 00:00:00 - 2021-12-23 00:00:00 n= 452
TEST : 2022-01-03 00:00:00 - 2024-03-04 00:00:00 n= 544


**Escalado** (Fit solo al train real)

In [11]:

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)


**Preparación de secuencias (LSTM)**

In [12]:
def make_sequences(X: np.ndarray, y: np.ndarray, lookback: int):
    X_seq, y_seq = [], []
    for t in range(lookback - 1, len(X)):
        X_seq.append(X[t - lookback + 1:t + 1])
        y_seq.append(y[t])
    return np.asarray(X_seq, np.float32), np.asarray(y_seq, np.int32)

In [13]:
lookback = 10  
X_tr_seq, y_tr_seq = make_sequences(X_tr_scaled, y_tr.values, lookback)
X_val_seq, y_val_seq = make_sequences(X_val_scaled, y_val.values, lookback)
X_test_seq, y_test_seq = make_sequences(X_test_scaled, y_test.values, lookback)

In [14]:
print(f"Entrenamiento: {X_tr_seq.shape}")
print(f"Prueba (validación): {X_val_seq.shape}") 
print(f"Prueba (Test): {X_test_seq.shape}")

print("X_tr_seq:", X_tr_seq.shape, "y_train_seq:", y_tr_seq.shape)
print("X_val_seq :", X_val_seq.shape,  "y_val_seq :", y_val_seq.shape)
print("X_test_seq :", X_test_seq.shape,  "y_test_seq :", y_test_seq.shape)

test_dates_seq = X_test_raw.index[lookback-1:]
print("Fechas test (seq):", test_dates_seq.min(), "->", test_dates_seq.max())

Entrenamiento: (1796, 10, 18)
Prueba (validación): (443, 10, 18)
Prueba (Test): (535, 10, 18)
X_tr_seq: (1796, 10, 18) y_train_seq: (1796,)
X_val_seq : (443, 10, 18) y_val_seq : (443,)
X_test_seq : (535, 10, 18) y_test_seq : (535,)
Fechas test (seq): 2022-01-14 00:00:00 -> 2024-03-04 00:00:00


**Ajuste de hiperparámetros (grid reducido) y entreno**
- Se compara un conjunto reducido de tamaños de red (nº de neuronas: 32 y 64) para controlar la complejidad del modelo.
- Se mantiene el resto de parámetros fijos.

In [15]:
n_features = X_tr_seq.shape[-1]

resultados_totales = []
neuronas_list = [32, 64] 

for neuronas in neuronas_list:
    print (f"Entrenando modelo {MODELO_BASE} {HORIZONTE} {SENTIMIENTO} sentimiento con {neuronas} neuronas...")
    model = models.Sequential([
        layers.Input(shape=(lookback, n_features)),
        layers.LSTM(neuronas, return_sequences=False,kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(neuronas//2, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="roc_auc", curve="ROC")])
    
    callbacks = [
        EarlyStopping(monitor="val_roc_auc", mode="max",patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_roc_auc", mode="max", factor=0.5, patience=5, min_lr=1e-5, verbose=1,)
    ]
  
    
    history = model.fit(
        X_tr_seq, y_tr_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=100,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )
    
    y_proba = model.predict(X_test_seq, verbose=0).ravel()
    y_pred  = (y_proba >= 0.5).astype(int)  
    

    metrics = {
        "Familia": FAMILIA,
        "Modelo": f"{MODELO_BASE}_{neuronas}",
        "Horizonte": HORIZONTE,
        "Sentimiento" : SENTIMIENTO,
        "Acc": accuracy_score(y_test_seq, y_pred),
        "B_Acc": balanced_accuracy_score(y_test_seq, y_pred),
        "Precision": precision_score(y_test_seq, y_pred, zero_division=0),
        "Recall": recall_score(y_test_seq, y_pred, zero_division=0),
        "F1": f1_score(y_test_seq, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test_seq, y_proba),
        "Conf_Matrix": confusion_matrix(y_test_seq, y_pred),
    }





    resultados_totales.append(metrics)


Entrenando modelo LSTM 5d FULL sentimiento con 32 neuronas...
Epoch 1/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.6008 - loss: 0.6750 - roc_auc: 0.5361 - val_accuracy: 0.6524 - val_loss: 0.6491 - val_roc_auc: 0.5397 - learning_rate: 0.0010
Epoch 2/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.6141 - loss: 0.6641 - roc_auc: 0.5689 - val_accuracy: 0.6524 - val_loss: 0.6478 - val_roc_auc: 0.5531 - learning_rate: 0.0010
Epoch 3/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6231 - loss: 0.6611 - roc_auc: 0.5854 - val_accuracy: 0.6524 - val_loss: 0.6492 - val_roc_auc: 0.5523 - learning_rate: 0.0010
Epoch 4/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6297 - loss: 0.6545 - roc_auc: 0.6039 - val_accuracy: 0.6524 - val_loss: 0.6514 - val_roc_auc: 0.5604 - learning_rate: 0.0010
Epoch 5/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6308 - loss: 0.6409 - roc_auc: 0.6487 - val_accuracy: 0.6524 - val_loss: 0.6583 - val_roc_auc: 0.5528 - le

**Guardado de resultados**
- Exportación a CSV para combinar en Notebook de resultados `NB05a_Resultados.ipynb`


In [16]:
os.makedirs(OUT_DIR, exist_ok=True)

df_res = pd.DataFrame(resultados_totales)
df_res["Conf_Matrix"] = df_res["Conf_Matrix"].astype(str)
df_res.to_csv(OUT_PATH, index=False)
if os.path.exists(OUT_PATH): 
    print(f"Archivo guardado correctamente en {OUT_PATH}") 
else: 
    print("Error: el archivo no se ha guardado.")
df_res

Archivo guardado correctamente en ../../resultados/resultados_lstm_5d_SENT_FULL.csv


,Familia,Modelo,Horizonte,Sentimiento,Acc,B_Acc,Precision,Recall,F1,ROC_AUC,Conf_Matrix
0,DL,LSTM_32,5d,FULL,0.564486,0.5,0.564486,1.0,0.721625,0.538044,[[ 0 233]\n [ 0 302]]
1,DL,LSTM_64,5d,FULL,0.564486,0.5,0.564486,1.0,0.721625,0.532246,[[ 0 233]\n [ 0 302]]
